#### Data explorer
The following notebook is for exploring Snakemake produced data inputs for PyPSA for final runs

In [ ]:
import pandas as pd
import geopandas as gpd 
import xarray as xr

from pathlib import Path

import yaml

# Read YAML file
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)


In [ ]:
n_path = "./networks/ME IRP 2024/elec/capacity-IRP_REF_CI.nc"
base_path  = "./networks/ME IRP 2024/base/capacity-IRP_REF_CI.nc"

In [ ]:
ds_base = xr.open_dataset(base_path)
ds_network = xr.open_dataset(n_path)

In [ ]:

filepath = "~/showcase/pypsa-rsa/pre_processing/resource_processing/renewable_profiles_updated.nc"

In [ ]:
REGIONAL_MAPPING = {
    'Eastern Cape': 'EC',
    'Free State': 'FS',
    'Gauteng': 'GP',
    'Hydra Central': 'HY',
    'KwaZulu Natal': 'ZN',
    'Limpopo':'LP',
    'Mpumalanga': 'MP',
    'North West':  'NW',
    'Northern Cape': 'NC',
    'Western Cape':  'WC'
}

# Easy inverses and indices using dict comprehension
CODE_TO_NAME = {v: k for k, v in REGIONAL_MAPPING.items()}
# Position-based indices (order matters, so use sorted for consistency)
REGION_NAMES = sorted(REGIONAL_MAPPING.keys())  # Ensures consistent ordering
NAME_TO_INDEX = {name: i for i, name in enumerate(REGION_NAMES)}
INDEX_TO_NAME = {i: name for i, name in enumerate(REGION_NAMES)}

In [ ]:
# Easy inverses and indices using dict comprehension
CODE_TO_NAME = {v: k for k, v in REGIONAL_MAPPING.items()}

# Position-based indices (order matters, so use sorted for consistency)
REGION_NAMES = sorted(REGIONAL_MAPPING.keys())  # Ensures consistent ordering
NAME_TO_INDEX = {name: i for i, name in enumerate(REGION_NAMES)}
INDEX_TO_NAME = {i: name for i, name in enumerate(REGION_NAMES)}

# Or if you want to preserve the original dict order (Python 3.7+):
# NAME_TO_INDEX = {name: i for i, name in enumerate(REGIONAL_MAPPING.keys())}
# INDEX_TO_NAME = {i: name for i, name in enumerate(REGIONAL_MAPPING.keys())}

# Convenience mappings for different suffix types
CODE_TO_INDEX = {REGIONAL_MAPPING[name]: NAME_TO_INDEX[name] for name in REGION_NAMES}
INDEX_TO_CODE = {NAME_TO_INDEX[name]: REGIONAL_MAPPING[name] for name in REGION_NAMES}

In [ ]:
NAME_TO_INDEX

In [ ]:
# This should work (from your investigation)
wind_data = xr.open_dataarray(filepath, group="wind_10_era5")
print(f"Wind data: {wind_data.dims}, shape {wind_data.shape}")
print(f"Bus coords: {wind_data.coords['bus'].values}")
print(f"Time range: {wind_data.coords['time'].values[[0, -1]]}")

# Test solar
solar_data = xr.open_dataarray(filepath, group="solar_pv_10_sarah")
print(f"Solar data: {solar_data.dims}, shape {solar_data.shape}")

wind_data.close()
solar_data.close()

In [ ]:
# Explore with PyPSA
import pypsa

In [ ]:
supply_regions = Path(config["gis"]["path"] + "/supply_regions/rsa_supply_regions.gpkg").expanduser()
supply_regions

In [ ]:
 regions = gpd.read_file(
       supply_regions,
        layer='10',
    )


In [ ]:
regions.name.to_list()

In [ ]:
n = pypsa.Network(n_path)

In [ ]:
# Network summary
print(f"Network Name: {n.name}")
print(f"Multi-investment: {n.multi_invest}")
print(f"Investment Periods: {n.investment_periods if n.multi_invest else 'Single period'}")
print(f"Snapshots: {len(n.snapshots)} timesteps")
print(f"Start: {n.snapshots[0]}")
print(f"End: {n.snapshots[-1]}")

# Network size
print(f"\nNetwork Components:")
print(f"Buses: {len(n.buses)}")
print(f"Generators: {len(n.generators)}")
print(f"Storage Units: {len(n.storage_units)}")
print(f"Loads: {len(n.loads)}")
print(f"Lines: {len(n.lines)}")
print(f"Links: {len(n.links)}")
print(f"Carriers: {len(n.carriers)}")

Other stuff.... TSAM use

In [ ]:
import tsam.timeseriesaggregation as tsam

def apply_time_segmentation(n, segments, config):
    logging.info(f"Aggregating time series to {segments} segments.")    
    years = n.investment_periods if n.multi_invest else [n.snapshots[0].year]

    if len(years) == 1:
        segmented_df, weightings = single_year_segmentation(n, n.snapshots, segments, config)
    else:

        with ProcessPoolExecutor(max_workers = min(len(years),config['nprocesses'])) as executor:
            parallel_seg = {
                year: executor.submit(
                    single_year_segmentation,
                    n,
                    n.snapshots[n.snapshots.get_level_values(0) == year],
                    segments,
                    config
                )
                for year in years
            }

        segmented_df = pd.concat(
            [parallel_seg[year].result()[0] for year in parallel_seg], axis=0
        )
        weightings = pd.concat(
            [parallel_seg[year].result()[1] for year in parallel_seg], axis=0
        )

    n.set_snapshots(segmented_df.index)
    n.snapshot_weightings = weightings   
    
    assign_segmented_df_to_network(segmented_df, "_load", "", n.loads_t.p_set)
    assign_segmented_df_to_network(segmented_df, "_max", "", n.generators_t.p_max_pu)
    assign_segmented_df_to_network(segmented_df, "_min", "", n.generators_t.p_min_pu)
    assign_segmented_df_to_network(segmented_df, "_inflow", "", n.storage_units_t.inflow)

    return n

In [ ]:
# n = apply_time_segmentation(n, m[0][:-3], config["tsam_clustering"])